# Inflacao, juros e cambio no Brasil

Esta analise combina series do Sistema Gerenciador de Series Temporais (SGS) do Banco Central: IPCA mensal (codigo 433, %), Selic anual (codigo 12, % a.a.) e cambio do dolar (codigo 1, BRL por USD). Os dados sao lidos do PostgreSQL por views SQL.

Vamos observar a inflacao acumulada em 12 meses, estimar o juro real pela equacao de Fisher e explorar a associacao entre variacao mensal do cambio e IPCA. Correlacao nao demonstra causalidade.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

ROOT = Path.cwd()
if not (ROOT / "sql" / "02_views.sql").exists():
    ROOT = next(path for path in ROOT.parents if (path / "sql" / "02_views.sql").exists())
sys.path.insert(0, str(ROOT))

from src.config import criar_engine

engine = criar_engine()
try:
    sql_views = (ROOT / "sql" / "02_views.sql").read_text(encoding="utf-8")
    with engine.begin() as conexao:
        conexao.exec_driver_sql(sql_views)

    ipca_12m = pd.read_sql_query(
        text("SELECT data, inflacao_12_meses_pct FROM vw_ipca_12_meses ORDER BY data"), engine
    )
    juros_reais = pd.read_sql_query(
        text("SELECT data, selic_anual_pct, inflacao_12_meses_pct, juro_real_pct FROM vw_juro_real ORDER BY data"), engine
    )
    mensal = pd.read_sql_query(
        text("SELECT data, cambio_medio_brl_por_usd, variacao_cambio_mensal_pct, ipca_mensal_pct, juro_real_pct FROM vw_analise_mensal ORDER BY data"), engine
    )
finally:
    engine.dispose()

for tabela in (ipca_12m, juros_reais, mensal):
    tabela["data"] = pd.to_datetime(tabela["data"])

print(f"IPCA 12 meses: {len(ipca_12m):,} meses; juro real: {len(juros_reais):,} meses; analise mensal: {len(mensal):,} meses.")
display(ipca_12m.tail())
display(juros_reais.tail())

## Validacao das metricas

A inflacao acumulada e o produto das variacoes mensais (1 + IPCA/100), menos 1. Com dados carregados, comparamos dezembro de 2023 com os **4,62%** do IPCA oficial divulgado pelo IBGE. O juro real usa taxas anuais na mesma unidade percentual: (1 + Selic/100) / (1 + inflacao 12m/100) - 1. A comparacao de dezembro de 2023 usa o acumulado dos 12 meses anteriores. O exemplo de Fisher abaixo e uma verificacao aritmetica.

In [ ]:
selic_exemplo_pct = 10.0
inflacao_exemplo_pct = 4.0
juro_real_exemplo_pct = ((1 + selic_exemplo_pct / 100) / (1 + inflacao_exemplo_pct / 100) - 1) * 100
assert np.isclose(juro_real_exemplo_pct, 5.76923076923077)
print(f"Fisher (10% nominal, 4% inflacao): {juro_real_exemplo_pct:.4f}% real.")

referencia_ipca = pd.Timestamp("2023-12-01")
resultado_2023 = ipca_12m.loc[ipca_12m["data"].eq(referencia_ipca), "inflacao_12_meses_pct"]
if not resultado_2023.empty:
    ipca_calculado = float(resultado_2023.iloc[0])
    ipca_oficial = 4.62
    diferenca_pp = ipca_calculado - ipca_oficial
    print(f"IPCA 12m em dez/2023: {ipca_calculado:.2f}% (IBGE: {ipca_oficial:.2f}%; diferenca: {diferenca_pp:+.2f} p.p.).")
    assert abs(diferenca_pp) <= 0.02, "O acumulado diverge do IPCA oficial de 2023."
else:
    print("Validacao oficial do IPCA 2023 pendente: a view nao contem dez/2023; carregue a serie SGS 433 desde 2023.")

## Associacao entre cambio e inflacao

Usamos a variacao percentual mensal da media do cambio (BRL por USD) e o IPCA mensal, nao os niveis das series. A correlacao de Pearson resume associacao linear no periodo observado, mas nao controla por defasagens, politica monetaria, choques externos ou outros fatores. Associacao nao implica causalidade.

In [ ]:
pares = mensal[["variacao_cambio_mensal_pct", "ipca_mensal_pct"]].dropna()
if len(pares) >= 2 and pares.std(ddof=0).gt(0).all():
    correlacao = float(np.corrcoef(
        pares["variacao_cambio_mensal_pct"].to_numpy(),
        pares["ipca_mensal_pct"].to_numpy(),
    )[0, 1])
    print(f"Correlacao de Pearson cambio x IPCA mensal: {correlacao:.3f} (n={len(pares)} meses).")
else:
    correlacao = np.nan
    print(f"Correlacao nao calculada: sao necessarios dois pares variaveis; disponiveis {len(pares)}.")
if not mensal.empty:
    display(mensal.tail())

## Evolucao das series

Os paineis mantem escalas e unidades explicitas. O primeiro compara taxas percentuais anuais; o segundo mostra o cambio em BRL por USD; o terceiro relaciona variacoes mensais percentuais.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14), constrained_layout=True)
if not ipca_12m.empty:
    axes[0].plot(ipca_12m["data"], ipca_12m["inflacao_12_meses_pct"], label="IPCA acumulado em 12 meses")
if not juros_reais.empty:
    axes[0].plot(juros_reais["data"], juros_reais["selic_anual_pct"], label="Selic anual (ultimo valor do mes)")
    axes[0].plot(juros_reais["data"], juros_reais["juro_real_pct"], label="Juro real ex-post (Fisher)")
axes[0].set_title("Inflacao, Selic e juro real")
axes[0].set_ylabel("Taxa (% ao ano)")
axes[0].set_xlabel("Data")
if axes[0].lines:
    axes[0].legend()
axes[0].grid(True, alpha=0.3)

if not mensal.empty:
    axes[1].plot(mensal["data"], mensal["cambio_medio_brl_por_usd"], color="tab:green")
axes[1].set_title("Cambio medio mensal")
axes[1].set_ylabel("BRL por USD")
axes[1].set_xlabel("Data")
axes[1].grid(True, alpha=0.3)

if not pares.empty:
    axes[2].scatter(pares["variacao_cambio_mensal_pct"], pares["ipca_mensal_pct"], alpha=0.7)
axes[2].set_title("Variacao mensal do cambio x IPCA mensal")
axes[2].set_xlabel("Variacao do cambio medio (%)")
axes[2].set_ylabel("IPCA mensal (%)")
axes[2].grid(True, alpha=0.3)
plt.show()

## Conclusoes

Os achados abaixo sao calculados a partir dos dados disponiveis no banco. Sem carga historica, o notebook registra essa limitacao em vez de apresentar conclusoes inventadas.

In [ ]:
achados = []
if not ipca_12m.empty:
    ultimo_ipca = ipca_12m.iloc[-1]
    achados.append(f"Em {ultimo_ipca['data']:%m/%Y}, o IPCA acumulado em 12 meses foi {ultimo_ipca['inflacao_12_meses_pct']:.2f}%.")
if not juros_reais.empty:
    ultimo_juro = juros_reais.iloc[-1]
    achados.append(f"Em {ultimo_juro['data']:%m/%Y}, a Selic anual foi {ultimo_juro['selic_anual_pct']:.2f}% e o juro real ex-post estimado foi {ultimo_juro['juro_real_pct']:.2f}%.")
if not mensal.empty:
    ultimo_cambio = mensal.dropna(subset=["cambio_medio_brl_por_usd"]).iloc[-1]
    achados.append(f"Em {ultimo_cambio['data']:%m/%Y}, o cambio medio foi {ultimo_cambio['cambio_medio_brl_por_usd']:.2f} BRL/USD.")
if np.isfinite(correlacao):
    achados.append(f"No periodo pareado, a correlacao cambio x IPCA foi {correlacao:.3f} (n={len(pares)}); isso nao estabelece causalidade.")
if not achados:
    achados.append("O banco nao contem observacoes; as metricas e correlacoes precisam de dados carregados para gerar achados empiricos.")
if len(achados) < 3:
    achados.append("A validacao do IPCA oficial de 2023 exige os 12 meses da serie SGS 433 carregados no banco.")
if len(achados) < 3:
    achados.append("As series disponiveis sao insuficientes para interpretacao empirica; carregue os dados e execute novamente o notebook.")
for numero, achado in enumerate(achados[:5], start=1):
    print(f"{numero}. {achado}")
print()
print("Limitacao: correlacao contemporanea nao identifica causalidade nem efeitos defasados.")

### Fontes e convencoes

- BCB/SGS: series 433 (IPCA mensal), 12 (Selic anual) e 1 (cambio). Os dados dependem do periodo carregado no PostgreSQL.
- IBGE, IPCA de dezembro de 2023: [IPCA vai a 0,56% em dezembro e fecha o ano em 4,62%](https://agenciadenoticias.ibge.gov.br/agencia-sala-de-imprensa/2013-agencia-de-noticias/releases/38984-ipca-vai-a-0-56-em-dezembro-e-fecha-o-ano-em-4-62).
- O juro real e uma aproximacao ex-post com a ultima Selic anual observada no mes, comparada ao IPCA acumulado em 12 meses; nao e a taxa real esperada pelo mercado.